# Credit Card Default Prediction — Training Notebook

**Stage 1.5** of the project (see `docs/progress.md`). Run this in Google Colab.

Dataset: [Default of Credit Card Clients](https://www.kaggle.com/datasets/uciml/default-of-credit-card-clients-dataset)

Workflow reminder (full detail in `docs/mlflow-workflow.md`):
1. Everything here logs to a **local** MLflow tracking URI (`file:./mlruns`) — no network setup needed from Colab.
2. At the end of the session, zip `mlruns/` and download it, along with the best model's artifacts.
3. Drop `mlruns/` into `infra/mlflow/data/mlruns` on your machine to browse everything in the local MLflow UI.
4. Drop the exported model + preprocessing artifacts into `ml/artifacts/` for the FastAPI backend (Stage 2).

## 0. Setup

In [ ]:
!pip install -q mlflow torch scikit-learn imbalanced-learn pandas matplotlib seaborn optuna

import mlflow

# Local file-store tracking — matches the FileStore backend used by the
# Dockerized MLflow server, so mlruns/ can be copied over directly later.
mlflow.set_tracking_uri("file:./mlruns")
mlflow.set_experiment("credit-card-default")

## 1. Load Data

Download the dataset from Kaggle (via `kagglehub` or manual upload) into `ml/data/` conventions — in Colab, just load it into a DataFrame directly.

In [ ]:
# TODO: load raw CSV into a DataFrame

## 2. Exploratory Data Analysis

- Target distribution (`default.payment.next.month`) — quantify the class imbalance
- Univariate distributions: `LIMIT_BAL`, `AGE`, `BILL_AMT1-6`, `PAY_AMT1-6`
- `PAY_0..PAY_6` repayment status patterns vs. default
- Data quality: undocumented codes in `EDUCATION` (0, 5, 6) and `MARRIAGE` (0)
- Correlation heatmap

In [ ]:
# TODO: EDA

## 3. Preprocessing & Feature Engineering

- Consolidate undocumented `EDUCATION`/`MARRIAGE` categories
- Encode categoricals (`SEX`, `EDUCATION`, `MARRIAGE`)
- Engineered features: payment-to-bill ratios, delinquency streak length across `PAY_0..PAY_6`, spending trend slope across `BILL_AMT1..6`
- Scale numeric features (fit on train only — `StandardScaler`)
- Stratified train / validation / test split (stratify on target, given the imbalance)
- Persist the fitted scaler + feature column order — needed later for backend inference

In [ ]:
# TODO: preprocessing + feature engineering

## 4. Handling Class Imbalance

Compare, as separate logged MLflow runs:
- Baseline (no handling)
- Class-weighted loss (`pos_weight` in `BCEWithLogitsLoss`)
- SMOTE oversampling (train split only)
- Random undersampling (train split only)

Evaluate on precision / recall / F1 / PR-AUC — **not** accuracy alone.

In [ ]:
# TODO: imbalance strategy comparison

## 5. MLP Model (PyTorch)

Define the architecture once, parametrized (hidden layer sizes, dropout rate) so it can be reused across the
hyperparameter search below. Keep `model_config.json`-style parameters explicit — the backend will need to
reconstruct this exact architecture at inference time.

In [ ]:
# TODO: nn.Module definition

## 6. Weight Initialization Experiments

Compare zero init (baseline, expect it to fail/stagnate), Xavier/Glorot, and He initialization. Log each as a run.

In [ ]:
# TODO: weight init comparison

## 7. Regularization Experiments

Dropout, Batch Normalization, L2 weight decay, Early Stopping — ablate individually and combined, log each combination.

In [ ]:
# TODO: regularization ablations

## 8. Hyperparameter Tuning

- Grid search and random search over: learning rate, batch size, hidden layer sizes, dropout rate, optimizer (SGD / Momentum / RMSProp / Adam / AdamW)
- Optional stretch: Optuna study for a more efficient search
- Log every trial as an MLflow run (params + metrics), so the comparison is queryable later

In [ ]:
# TODO: hyperparameter search

## 9. Final Model Evaluation & Error Analysis

- Select best run from MLflow (`mlflow.search_runs`, sort by chosen metric — decide precision vs. recall priority first)
- Confusion matrix, classification report, ROC and PR curves on the held-out test set
- Inspect misclassified examples — any pattern (e.g. borderline `PAY_0` values)?

In [ ]:
# TODO: final evaluation

## 10. Export Artifacts

Save, then download and place into `ml/artifacts/`:
- `model/model.pt` — trained weights
- `model/model_config.json` — architecture params needed to reconstruct the `nn.Module`
- `preprocessing/scaler.pkl` — fitted scaler
- `preprocessing/feature_columns.json` — exact column order/names expected at inference
- `metrics/evaluation_report.json` — final metrics for the README/docs

In [ ]:
# TODO: export artifacts + zip mlruns/ for download